# Instagram Influencer Analytics Dashboard

Dashboard interaktif untuk menganalisis akun Instagram populer berdasarkan dataset yang diberikan.

**File dataset:** `instagram_influencers_dataset.csv`  
**Sumber awal:** `DOC-20261024-WA0026.pdf`  
**Cara pakai:** simpan file notebook dan CSV di folder yang sama, lalu jalankan semua cell di VS Code dengan Jupyter extension.

> Catatan: data diekstrak dari PDF. Periksa kembali beberapa nama akun dan nilai yang kosong karena tata letak PDF dapat menyebabkan teks kolom menyatu saat diekstrak. Angka metrik dengan k/m/b dikonversi ke angka penuh untuk perhitungan.


In [1]:
%pip install pandas plotly ipywidgets

  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached ipywidgets-8.1.9-py3-none-any.whl.metadata (2.4 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached widgetsnbextension-4.0.16-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl.metadata (20 kB)
Using cached plotly-7.1.0-py3-none-any.whl (9.7 MB)
Using cached ipywidgets-8.1.9-py3-none-any.whl (140 kB)
Using cached jupyterlab_widgets-3.0.17-py3-none-any.whl (217 kB)
Using cached widgetsnbextension-4.0.16-py3-none-any.whl (2.2 MB)
Using cached narwhals-2.27.1-py3-none-any.whl (483 kB)

   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   -------- ------------------------------- 1/5 [narwhals]
   ---


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

DATA_FILE = "instagram_influencers_dataset.csv"
df = pd.read_csv(DATA_FILE)

# Konversi metrik yang dipakai dalam dashboard
numeric_cols = [
    "influence_score_num", "posts_num", "followers_num", "avg_likes_num",
    "engagement_rate_num", "new_post_avg_like_num", "total_likes_num"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["channel_info"] = df["channel_info"].fillna("Unknown").astype(str)
print(f"Dataset berhasil dimuat: {len(df):,} baris")
display(df.head())


FileNotFoundError: [Errno 2] No such file or directory: 'instagram_influencers_dataset.csv'

## Dashboard interaktif
Pilih filter, lalu klik **Tampilkan dashboard**. KPI, grafik, dan insight akan mengikuti data yang terpilih.

In [ ]:
account_search = widgets.Text(
    value="", placeholder="Cari nama akun...", description="Akun:",
    layout=widgets.Layout(width="320px")
)
min_score = widgets.IntSlider(
    value=0, min=0, max=100, step=1, description="Influence min:",
    continuous_update=False, layout=widgets.Layout(width="420px")
)
max_score = widgets.IntSlider(
    value=100, min=0, max=100, step=1, description="Influence max:",
    continuous_update=False, layout=widgets.Layout(width="420px")
)
metric_choice = widgets.Dropdown(
    options=[
        ("Followers", "followers_num"),
        ("Rata-rata likes", "avg_likes_num"),
        ("Total likes", "total_likes_num"),
        ("Rata-rata likes postingan baru", "new_post_avg_like_num"),
    ],
    value="followers_num", description="Metrik grafik:"
)
show_n = widgets.IntSlider(
    value=10, min=5, max=25, step=5, description="Top akun:",
    continuous_update=False, layout=widgets.Layout(width="320px")
)
run_button = widgets.Button(description="Tampilkan dashboard", button_style="primary", icon="refresh")
out = widgets.Output()

def human_num(x):
    if pd.isna(x): return "N/A"
    x = float(x)
    for unit, div in [("B",1e9),("M",1e6),("K",1e3)]:
        if abs(x) >= div: return f"{x/div:.2f}{unit}"
    return f"{x:,.0f}"

def render_dashboard(_=None):
    with out:
        clear_output(wait=True)
        data = df.copy()
        data = data[
            data["influence_score_num"].between(min_score.value, max_score.value, inclusive="both")
        ]
        query = account_search.value.strip().lower()
        if query:
            data = data[data["channel_info"].str.lower().str.contains(query, na=False)]
        if data.empty:
            display(Markdown("### Tidak ada data untuk filter ini. Coba ubah filter."))
            return

        total_accounts = data["channel_info"].nunique()
        total_followers = data["followers_num"].sum(min_count=1)
        avg_engagement = data["engagement_rate_num"].mean()
        avg_influence = data["influence_score_num"].mean()

        kpi_html = f'''
        <div style="display:flex;flex-wrap:wrap;gap:12px">
          <div style="padding:16px;border-radius:12px;background:#eef2ff;min-width:150px"><b>Jumlah akun</b><h2>{total_accounts:,}</h2></div>
          <div style="padding:16px;border-radius:12px;background:#ecfeff;min-width:150px"><b>Total followers</b><h2>{human_num(total_followers)}</h2></div>
          <div style="padding:16px;border-radius:12px;background:#f0fdf4;min-width:150px"><b>Rata-rata engagement</b><h2>{avg_engagement:.2f}%</h2></div>
          <div style="padding:16px;border-radius:12px;background:#fff7ed;min-width:150px"><b>Rata-rata influence</b><h2>{avg_influence:.1f}</h2></div>
        </div>'''
        display(widgets.HTML(value=kpi_html))

        plot_data = data.dropna(subset=[metric_choice.value]).nlargest(show_n.value, metric_choice.value)
        if not plot_data.empty:
            fig = px.bar(
                plot_data.sort_values(metric_choice.value, ascending=True),
                x=metric_choice.value, y="channel_info", orientation="h",
                title=f"Top akun berdasarkan {metric_choice.label}",
                labels={metric_choice.value: metric_choice.label, "channel_info":"Nama akun"},
                hover_data=["influence_score_num", "engagement_rate_num", "followers_num"]
            )
            fig.update_layout(height=max(420, len(plot_data)*28), yaxis_title="")
            display(fig)

        engagement_data = data.dropna(subset=["engagement_rate_num"]).nlargest(show_n.value, "engagement_rate_num")
        if not engagement_data.empty:
            fig2 = px.bar(
                engagement_data.sort_values("engagement_rate_num", ascending=True),
                x="engagement_rate_num", y="channel_info", orientation="h",
                title="Akun dengan engagement rate tertinggi",
                labels={"engagement_rate_num":"Engagement rate (%)","channel_info":"Nama akun"}
            )
            fig2.update_layout(height=max(420, len(engagement_data)*28), yaxis_title="")
            display(fig2)

        # Insight otomatis berdasarkan subset terpilih
        insights = []
        valid_eng = data.dropna(subset=["engagement_rate_num"])
        valid_followers = data.dropna(subset=["followers_num"])
        if not valid_eng.empty:
            best = valid_eng.loc[valid_eng["engagement_rate_num"].idxmax()]
            insights.append(f"Engagement rate tertinggi pada data terfilter dimiliki @{best['channel_info']} ({best['engagement_rate_num']:.2f}%).")
        if not valid_followers.empty:
            biggest = valid_followers.loc[valid_followers["followers_num"].idxmax()]
            insights.append(f"Jumlah followers terbesar pada data terfilter dimiliki @{biggest['channel_info']} ({human_num(biggest['followers_num'])}).")
        if not valid_eng.empty and not valid_followers.empty:
            corr_data = data[["followers_num","engagement_rate_num"]].dropna()
            if len(corr_data) >= 3:
                corr = corr_data["followers_num"].corr(corr_data["engagement_rate_num"])
                if pd.notna(corr):
                    label = "positif" if corr > 0.1 else "negatif" if corr < -0.1 else "lemah/hampir tidak ada"
                    insights.append(f"Korelasi followers dan engagement rate pada data terfilter tergolong {label} (r = {corr:.2f}); followers besar tidak otomatis berarti engagement tinggi.")
        display(Markdown("### 3 Insight utama\n" + "\n".join(f"{i+1}. {s}" for i,s in enumerate(insights[:3]))))

        recommendation = "Prioritaskan evaluasi akun berdasarkan engagement rate dan rata-rata likes, bukan jumlah followers saja; validasi juga kualitas audiens sebelum memilih akun untuk kolaborasi."
        display(Markdown("### Rekomendasi\n" + recommendation))

        display(Markdown("### Pratinjau data terfilter"))
        display(data[["rank","channel_info","influence_score_num","followers_num","avg_likes_num","engagement_rate_num","total_likes_num"]].head(20))

run_button.on_click(render_dashboard)
display(widgets.VBox([
    widgets.HBox([account_search, show_n]),
    min_score, max_score,
    metric_choice,
    run_button,
    out
]))
render_dashboard()


## Tentang sumber dataset
Dashboard menggunakan file `instagram_influencers_dataset.csv`, yang dibuat dari tabel akun pada PDF sumber. File sumber menampilkan metrik seperti followers, rata-rata likes, engagement rate 60 hari, dan total likes. Negara tidak digunakan sebagai filter karena daftar negara pada PDF berada di halaman terpisah dan tidak dapat dipasangkan secara andal ke setiap akun melalui ekstraksi teks saja.